In [12]:
import json
import lzma
from pathlib import Path
import pandas as pd


def load_json_xz(path):
    with lzma.open(path, "rt", encoding="utf-8") as fh:
        return pd.DataFrame(json.load(fh))


files = sorted(Path("data").glob("*.json.xz"))
df = pd.concat((load_json_xz(path) for path in files), ignore_index=True)
df.shape

(70974, 169)

In [14]:
def extract_ticker_prices(row):
    tickers = row.get('mentioned_companies')
    if not isinstance(tickers, list) or len(tickers) == 0:
        return []

    ticker_rows = []
    for ticker in tickers:
        prev_price = row.get(f'prev_day_price_{ticker}')
        curr_price = row.get(f'curr_day_price_{ticker}')
        next_price = row.get(f'next_day_price_{ticker}')

        # Keep a ticker only when we can compute its movement target.
        if pd.notna(prev_price) and pd.notna(next_price):
            ticker_rows.append({
                'ticker': ticker,
                'prev_price': prev_price,
                'curr_price': curr_price,
                'next_price': next_price,
            })

    return ticker_rows

In [32]:
keep_cols = ["title", "maintext", "date_publish", "mentioned_companies", "sentiment", "emotion", "named_entities"]

# Keep the original article index after exploding so rows remain aligned.
ticker_data = df.apply(extract_ticker_prices, axis=1).explode().dropna()
ticker_df = pd.DataFrame(ticker_data.tolist(), index=ticker_data.index)
df_with_tickers = (
    df.loc[ticker_df.index, keep_cols]
    .reset_index(drop=True)
    .join(ticker_df.reset_index(drop=True))
)
df_with_tickers.shape

(85957, 11)

In [33]:
def clean_cols(df):
    df = df.copy()

    sentiment_df = pd.json_normalize(
        df["sentiment"].apply(lambda x: x if isinstance(x, dict) else {})
    ).add_prefix("raw_sentiment_")

    emotion_df = pd.json_normalize(
        df["emotion"].apply(lambda x: x if isinstance(x, dict) else {})
    ).add_prefix("raw_emotion_")

    df = pd.concat(
        [df.drop(columns=["sentiment", "emotion"]), sentiment_df, emotion_df],
        axis=1
    )

    for col in sentiment_df.columns.tolist() + emotion_df.columns.tolist():
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df

In [34]:
cleaned_df = clean_cols(df_with_tickers)
cleaned_df.head()

,title,maintext,date_publish,mentioned_companies,named_entities,ticker,prev_price,curr_price,next_price,raw_sentiment_negative,raw_sentiment_neutral,raw_sentiment_positive,raw_emotion_neutral,raw_emotion_surprise,raw_emotion_disgust,raw_emotion_anger,raw_emotion_sadness,raw_emotion_joy,raw_emotion_fear
0,LEAKED PHOTOS: Fitbit’s new headphones and tro...,Yahoo Finance has obtained photos of Fitbit’s ...,2017-05-01 14:04:36,[AAPL],"[{'entity_group': 'MISC', 'word': 'San Francis...",AAPL,143.64999,146.58000,147.50999,0.004726,0.994807,0.000467,0.959645,0.012228,0.008198,0.007327,0.006158,0.005158,0.001287
1,Testosterone is the enemy of smart investing d...,The pitfalls of the ‘hurry up’ hormone\nShutte...,2017-05-09 08:50:32,[MA],"[{'entity_group': 'MISC', 'word': 'Shut', 'nor...",MA,116.61000,116.41000,116.65000,0.002295,0.002935,0.994770,0.748777,0.181237,0.016509,0.018160,0.003979,0.012707,0.018630
2,Open-internet rules look dead. Now what?,President Trump’s new head of the Federal Comm...,2017-02-07 23:16:13,"[T, VZ]","[{'entity_group': 'PER', 'word': 'Trump', 'nor...",T,41.06000,41.12000,41.21000,0.031167,0.789774,0.179059,0.894539,0.045494,0.019059,0.024419,0.006296,0.004959,0.005234
3,Open-internet rules look dead. Now what?,President Trump’s new head of the Federal Comm...,2017-02-07 23:16:13,"[T, VZ]","[{'entity_group': 'PER', 'word': 'Trump', 'nor...",VZ,48.03000,48.04000,48.37000,0.031167,0.789774,0.179059,0.894539,0.045494,0.019059,0.024419,0.006296,0.004959,0.005234
4,A ruling against Google in Canada could affect...,The Supreme Court of Canada issued an order to...,2017-06-29 20:45:02,[GOOGL],"[{'entity_group': 'ORG', 'word': 'Google', 'no...",GOOGL,940.48999,917.78998,908.72998,0.039223,0.959608,0.001169,0.360192,0.005024,0.271656,0.305400,0.032139,0.001700,0.023889
